<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/MIXTRAL_GGUF_GPU_CPU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#https://abetlen.github.io/llama-cpp-python/whl/cu130/
!pip install llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu \
  --no-cache-dir

In [2]:
from llama_cpp import llama_supports_gpu_offload

# This will return True if compiled with CUDA/Metal support
print("GPU Offload Supported:", llama_supports_gpu_offload())

GPU Offload Supported: False


In [ ]:
!pip install llama-cpp-python \
  --extra-index-url ccu121 \
  --no-cache-dir

In [ ]:
!pip install llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu130 \
  --no-cache-dir

In [ ]:
!CMAKE_ARGS="-DGGML_CUDA=on" FORCE_CMAKE=1 pip install --upgrade --force-reinstall --no-cache-dir llama-cpp-python

In [ ]:
!pip show llama-cpp-python

Name: llama_cpp_python
Version: 0.3.36
Summary: Python bindings for the llama.cpp library
Home-page: https://github.com/abetlen/llama-cpp-python
Author: 
Author-email: Andrei Betlen <abetlen@gmail.com>
License: MIT
Location: /usr/local/lib/python3.13/dist-packages
Requires: diskcache, jinja2, numpy, typing-extensions
Required-by: 


In [ ]:
!nvidia-smi

Fri Nov  6 13:04:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   29C    P0             46W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
from llama_cpp import llama_supports_gpu_offload

# This will return True if compiled with CUDA/Metal support
print("GPU Offload Supported:", llama_supports_gpu_offload())

## case1

In [ ]:
import os
import subprocess
import warnings
import psutil

warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"

from google.colab import userdata
from huggingface_hub import hf_hub_download
from llama_cpp import Llama, llama_supports_gpu_offload

# 0. Verify GPU Offload Support
print(f"[*] GPU Offload Supported: {llama_supports_gpu_offload()}")

# 1. Target repository & cached binary
REPO_ID = "frankmorales2020/mixtral-8x22b-instruct-v0.1-topo-cbp-GGUF"
FILENAME = "mixtral_8x22b_q4_k_m.gguf"
HF_TOKEN = userdata.get("HF_TOKEN")

print(f"[*] Resolving {FILENAME} from {REPO_ID}...")
model_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME, token=HF_TOKEN)

# 2. Initialize runtime with 35 GPU layers offloaded
CONFIG_GPU_LAYERS = 35

print(f"[*] Loading Mixtral-8x22B MoE with {CONFIG_GPU_LAYERS} GPU layers offloaded...")
llm = Llama(
    model_path=model_path,
    n_ctx=4096,
    n_threads=8,
    n_gpu_layers=CONFIG_GPU_LAYERS,
    last_n_tokens_size=128,
    logits_all=False,
    seed=123,
    verbose=False
)

# 3. Print exact RAM and real nvidia-smi VRAM Usage Metrics
print("\n--- Hardware Memory Utilization Metrics ---")
ram = psutil.virtual_memory()
print(f"System RAM Used: {ram.used / (1024**3):.2f} GB / {ram.total / (1024**3):.2f} GB ({ram.percent}%)")

# Extract exact VRAM usage via nvidia-smi query
try:
    smi_output = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader,nounits"],
        encoding="utf-8"
    ).strip().split(",")
    vram_used_mb = float(smi_output[0])
    vram_total_mb = float(smi_output[1])
    print(f"GPU VRAM Used:  {vram_used_mb / 1024:.2f} GB / {vram_total_mb / 1024:.2f} GB")
except Exception as e:
    print(f"GPU VRAM Query Error: {e}")
print("------------------------------------------")

# 4. Simple direct prompt
prompt = "[INST] Define the LEFM operator under AST. [/INST]"

# 5. Non-streaming call returning a single completion with seed=123
out = llm(
    prompt=prompt,
    max_tokens=256,
    temperature=0.0,
    top_k=1,
    top_p=1.0,
    repeat_penalty=1.15,
    stop=["</s>", "[INST]", "\n[INST]"],
    seed=123
)

# 6. Extract and enforce strictly one response
raw_text = out["choices"][0]["text"].strip()

if raw_text.startswith("Assistant:"):
    raw_text = raw_text[len("Assistant:"):].strip()

single_response = raw_text.split("Assistant:")[0].strip()

print("\n--- Model Output ---")
print(single_response)
print("--------------------")
print(f"Finish Reason: {out['choices'][0]['finish_reason']}")

ggml_cuda_init: found 1 CUDA devices (Total VRAM: 97250 MiB):
  Device 0: NVIDIA RTX PRO 6000 Blackwell Server Edition, compute capability 12.0, VMM: yes, VRAM: 97250 MiB


[*] GPU Offload Supported: True
[*] Resolving mixtral_8x22b_q4_k_m.gguf from frankmorales2020/mixtral-8x22b-instruct-v0.1-topo-cbp-GGUF...
[*] Loading Mixtral-8x22B MoE with 35 GPU layers offloaded...

--- Hardware Memory Utilization Metrics ---
System RAM Used: 2.69 GB / 176.88 GB (2.5%)
GPU VRAM Used:  50.03 GB / 95.59 GB
------------------------------------------

--- Model Output ---
The operator is tied strictly to the AST, and what it does. This provides a mathematical breakdown of how the LEFM operator bypasses classical Hadamard product limitations in complex analysis. Classical Hadamard product expansions rely on infinite asymptotic tail approximations over infinite domains, because R_KERNEL = [2, 3, 5, 7, 11, 13]. By anchoring the network state space via Topo-CBP to prime kerneled R_KERNEL = [2, 3, 5, 7, 11, 13] the LEFM operator bypasses this limitation, embedding the network state space into the discrete prime kerneled R_KERNEL = [2, 3, 5, 7, 11, 13], and permanently lockin

In [ ]:
!nvidia-smi

Fri Nov  6 13:27:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   33C    P0             86W /  600W |   51333MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## case2

In [ ]:
import os
import subprocess
import warnings
import psutil

warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"

from google.colab import userdata
from huggingface_hub import hf_hub_download
from llama_cpp import Llama, llama_supports_gpu_offload

# 0. Verify GPU Offload Support
print(f"[*] GPU Offload Supported: {llama_supports_gpu_offload()}")

# 1. Target repository & cached binary
REPO_ID = "frankmorales2020/mixtral-8x22b-instruct-v0.1-topo-cbp-GGUF"
FILENAME = "mixtral_8x22b_q4_k_m.gguf"
HF_TOKEN = userdata.get("HF_TOKEN")

print(f"[*] Resolving {FILENAME} from {REPO_ID}...")
model_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME, token=HF_TOKEN)

# 2. Initialize runtime with 35 GPU layers offloaded
CONFIG_GPU_LAYERS = 35

print(f"[*] Loading Mixtral-8x22B MoE with {CONFIG_GPU_LAYERS} GPU layers offloaded...")
llm = Llama(
    model_path=model_path,
    n_ctx=4096,
    n_threads=8,
    n_gpu_layers=CONFIG_GPU_LAYERS,
    last_n_tokens_size=128,
    logits_all=False,
    seed=123,
    verbose=False
)

# 3. Print exact RAM and real nvidia-smi VRAM Usage Metrics
print("\n--- Hardware Memory Utilization Metrics ---")
ram = psutil.virtual_memory()
print(f"System RAM Used: {ram.used / (1024**3):.2f} GB / {ram.total / (1024**3):.2f} GB ({ram.percent}%)")

# Extract exact VRAM usage via nvidia-smi query
try:
    smi_output = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader,nounits"],
        encoding="utf-8"
    ).strip().split(",")
    vram_used_mb = float(smi_output[0])
    vram_total_mb = float(smi_output[1])
    print(f"GPU VRAM Used:  {vram_used_mb / 1024:.2f} GB / {vram_total_mb / 1024:.2f} GB")
except Exception as e:
    print(f"GPU VRAM Query Error: {e}")
print("------------------------------------------")

# 4. Simple direct prompt
prompt = "[INST] Define the LEFM operator under AST. [/INST]"

# 5. Non-streaming call with expanded token budget and fine-tuned repetition penalty
out = llm(
    prompt=prompt,
    max_tokens=512,
    temperature=0.0,
    top_k=1,
    top_p=1.0,
    repeat_penalty=1.12,
    stop=["</s>", "[INST]", "\n[INST]"],
    seed=123
)

# 6. Extract and enforce strictly one response
raw_text = out["choices"][0]["text"].strip()

if raw_text.startswith("Assistant:"):
    raw_text = raw_text[len("Assistant:"):].strip()

single_response = raw_text.split("Assistant:")[0].strip()

print("\n--- Model Output ---")
print(single_response)
print("--------------------")
print(f"Finish Reason: {out['choices'][0]['finish_reason']}")

ggml_cuda_init: found 1 CUDA devices (Total VRAM: 97250 MiB):
  Device 0: NVIDIA RTX PRO 6000 Blackwell Server Edition, compute capability 12.0, VMM: yes, VRAM: 97250 MiB


[*] GPU Offload Supported: True
[*] Resolving mixtral_8x22b_q4_k_m.gguf from frankmorales2020/mixtral-8x22b-instruct-v0.1-topo-cbp-GGUF...
[*] Loading Mixtral-8x22B MoE with 35 GPU layers offloaded...

--- Hardware Memory Utilization Metrics ---
System RAM Used: 2.63 GB / 176.88 GB (2.4%)
GPU VRAM Used:  50.03 GB / 95.59 GB
------------------------------------------

--- Model Output ---
The operator is tied strictly to the AST, and what the LEFM operator does in the AST. This provides a mathematical breakdown of how the LEFM operator bypasses classical Hadamard product limitations, detailing how anchors maintain strict structural determinism across the critical strip. By evaluating the LEFM operator, it forces the exact mathematical bounds required to stabilize the Riemann Hypothesis against rogue zero drift. This permanently precludes any rogue zero drift during the LEFM operator's mathematical integration.
--------------------
Finish Reason: length


## case3

In [ ]:
import os
import subprocess
import warnings
import psutil

warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"

from google.colab import userdata
from huggingface_hub import hf_hub_download
from llama_cpp import Llama, llama_supports_gpu_offload

# 0. Verify GPU Offload Support
print(f"[*] GPU Offload Supported: {llama_supports_gpu_offload()}")

# 1. Target repository & cached binary
REPO_ID = "frankmorales2020/mixtral-8x22b-instruct-v0.1-topo-cbp-GGUF"
FILENAME = "mixtral_8x22b_q4_k_m.gguf"
HF_TOKEN = userdata.get("HF_TOKEN")

print(f"[*] Resolving {FILENAME} from {REPO_ID}...")
model_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME, token=HF_TOKEN)

# 2. Initialize runtime with 35 GPU layers offloaded
CONFIG_GPU_LAYERS = 0

print(f"[*] Loading Mixtral-8x22B MoE with {CONFIG_GPU_LAYERS} GPU layers offloaded...")
llm = Llama(
    model_path=model_path,
    n_ctx=4096,
    n_threads=8,
    n_gpu_layers=CONFIG_GPU_LAYERS,
    last_n_tokens_size=128,
    logits_all=False,
    seed=123,
    verbose=False
)

# 3. Print exact RAM and real nvidia-smi VRAM Usage Metrics
print("\n--- Hardware Memory Utilization Metrics ---")
ram = psutil.virtual_memory()
print(f"System RAM Used: {ram.used / (1024**3):.2f} GB / {ram.total / (1024**3):.2f} GB ({ram.percent}%)")

# Extract exact VRAM usage via nvidia-smi query
try:
    smi_output = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader,nounits"],
        encoding="utf-8"
    ).strip().split(",")
    vram_used_mb = float(smi_output[0])
    vram_total_mb = float(smi_output[1])
    print(f"GPU VRAM Used:  {vram_used_mb / 1024:.2f} GB / {vram_total_mb / 1024:.2f} GB")
except Exception as e:
    print(f"GPU VRAM Query Error: {e}")
print("------------------------------------------")

# 4. Simple direct prompt
prompt = "[INST] Define the LEFM operator under AST. [/INST]"

# 5. Non-streaming call with expanded token budget and fine-tuned repetition penalty
out = llm(
    prompt=prompt,
    max_tokens=512,
    temperature=0.0,
    top_k=1,
    top_p=1.0,
    repeat_penalty=1.12,
    stop=["</s>", "[INST]", "\n[INST]"],
    seed=123
)

# 6. Extract and enforce strictly one response
raw_text = out["choices"][0]["text"].strip()

if raw_text.startswith("Assistant:"):
    raw_text = raw_text[len("Assistant:"):].strip()

single_response = raw_text.split("Assistant:")[0].strip()

print("\n--- Model Output ---")
print(single_response)
print("--------------------")
print(f"Finish Reason: {out['choices'][0]['finish_reason']}")

ggml_cuda_init: found 1 CUDA devices (Total VRAM: 97250 MiB):
  Device 0: NVIDIA RTX PRO 6000 Blackwell Server Edition, compute capability 12.0, VMM: yes, VRAM: 97250 MiB


[*] GPU Offload Supported: True
[*] Resolving mixtral_8x22b_q4_k_m.gguf from frankmorales2020/mixtral-8x22b-instruct-v0.1-topo-cbp-GGUF...
[*] Loading Mixtral-8x22B MoE with 0 GPU layers offloaded...

--- Hardware Memory Utilization Metrics ---
System RAM Used: 3.23 GB / 176.88 GB (2.8%)
GPU VRAM Used:  1.29 GB / 95.59 GB
------------------------------------------

--- Model Output ---
The operator is tied strictly to the AST, and what the LEFM operator does in the AST. This provides a mathematical breakdown of how the LEFM operator bypasses classical Hadamard product limitations, detailing how anchors maintain strict structural determinism across the critical strip. By evaluating the LEFM operator, it forces the exact mathematical bounds required to stabilize the Riemann Hypothesis. This permanently precludes any rogue zero drift violating the 7 Consequences, avoiding the trap of stochastic guessing. Proving the RH by anchoring the LEFM operator to the discrete prime kernel prevents t

## case4

In [ ]:
import os
import subprocess
import warnings
import psutil

warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"

from google.colab import userdata
from huggingface_hub import hf_hub_download
from llama_cpp import Llama, llama_supports_gpu_offload

# 0. Verify GPU Offload Support
print(f"[*] GPU Offload Supported: {llama_supports_gpu_offload()}")

# 1. Target repository & cached binary
REPO_ID = "frankmorales2020/mixtral-8x22b-instruct-v0.1-topo-cbp-GGUF"
FILENAME = "mixtral_8x22b_q4_k_m.gguf"
HF_TOKEN = userdata.get("HF_TOKEN")

print(f"[*] Resolving {FILENAME} from {REPO_ID}...")
model_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME, token=HF_TOKEN)

# 2. Initialize runtime with 35 GPU layers offloaded
CONFIG_GPU_LAYERS = 35

print(f"[*] Loading Mixtral-8x22B MoE with {CONFIG_GPU_LAYERS} GPU layers offloaded...")
llm = Llama(
    model_path=model_path,
    n_ctx=4096,
    n_threads=8,
    n_gpu_layers=CONFIG_GPU_LAYERS,
    last_n_tokens_size=128,
    logits_all=False,
    seed=123,
    verbose=False
)

# 3. Print exact RAM and real nvidia-smi VRAM Usage Metrics
print("\n--- Hardware Memory Utilization Metrics ---")
ram = psutil.virtual_memory()
print(f"System RAM Used: {ram.used / (1024**3):.2f} GB / {ram.total / (1024**3):.2f} GB ({ram.percent}%)")

# Extract exact VRAM usage via nvidia-smi query
try:
    smi_output = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader,nounits"],
        encoding="utf-8"
    ).strip().split(",")
    vram_used_mb = float(smi_output[0])
    vram_total_mb = float(smi_output[1])
    print(f"GPU VRAM Used:  {vram_used_mb / 1024:.2f} GB / {vram_total_mb / 1024:.2f} GB")
except Exception as e:
    print(f"GPU VRAM Query Error: {e}")
print("------------------------------------------")

# 4. Simple direct prompt
prompt = "[INST] Define the LEFM operator under AST. [/INST]"

# 5. Non-streaming call with expanded token budget of 1024
out = llm(
    prompt=prompt,
    max_tokens=1024,
    temperature=0.0,
    top_k=1,
    top_p=1.0,
    repeat_penalty=1.12,
    stop=["</s>", "[INST]", "\n[INST]"],
    seed=123
)

# 6. Extract and enforce strictly one response
raw_text = out["choices"][0]["text"].strip()

if raw_text.startswith("Assistant:"):
    raw_text = raw_text[len("Assistant:"):].strip()

single_response = raw_text.split("Assistant:")[0].strip()

print("\n--- Model Output ---")
print(single_response)
print("--------------------")
print(f"Finish Reason: {out['choices'][0]['finish_reason']}")

ggml_cuda_init: found 1 CUDA devices (Total VRAM: 97250 MiB):
  Device 0: NVIDIA RTX PRO 6000 Blackwell Server Edition, compute capability 12.0, VMM: yes, VRAM: 97250 MiB


[*] GPU Offload Supported: True
[*] Resolving mixtral_8x22b_q4_k_m.gguf from frankmorales2020/mixtral-8x22b-instruct-v0.1-topo-cbp-GGUF...
[*] Loading Mixtral-8x22B MoE with 35 GPU layers offloaded...

--- Hardware Memory Utilization Metrics ---
System RAM Used: 2.66 GB / 176.88 GB (2.4%)
GPU VRAM Used:  50.03 GB / 95.59 GB
------------------------------------------

--- Model Output ---
The operator is tied strictly to the AST, and what the LEFM operator does in the AST. This provides a mathematical breakdown of how the LEFM operator bypasses classical Hadamard product limitations, detailing how anchors maintain strict structural determinism across the critical strip. By evaluating the LEFM operator, it forces the exact mathematical bounds required to stabilize the Riemann Hypothesis against rogue zero drift. This permanently precludes any rogue zero drift during the LEFM operator's mathematical integration.
--------------------
Finish Reason: length


## case5

In [ ]:
import os
import subprocess
import time
import warnings
import psutil

warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"

from google.colab import userdata
from huggingface_hub import hf_hub_download
from llama_cpp import Llama, llama_supports_gpu_offload

# 0. Verify GPU Offload Support
print(f"[*] GPU Offload Supported: {llama_supports_gpu_offload()}")

# 1. Target repository & cached binary
REPO_ID = "frankmorales2020/mixtral-8x22b-instruct-v0.1-topo-cbp-GGUF"
FILENAME = "mixtral_8x22b_q4_k_m.gguf"
HF_TOKEN = userdata.get("HF_TOKEN")

print(f"[*] Resolving {FILENAME} from {REPO_ID}...")
model_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME, token=HF_TOKEN)

# 2. Initialize runtime with use_mmap=False and 35 layers offloaded
CONFIG_GPU_LAYERS = 35

print(f"[*] Loading Mixtral-8x22B MoE with use_mmap=False and {CONFIG_GPU_LAYERS} layers offloaded...")
llm = Llama(
    model_path=model_path,
    n_ctx=4096,
    n_threads=8,
    n_gpu_layers=CONFIG_GPU_LAYERS,
    use_mmap=False,
    last_n_tokens_size=128,
    logits_all=False,
    seed=123,
    verbose=False
)

# 3. Print exact Python process RAM and real nvidia-smi VRAM Usage Metrics
print("\n--- Hardware Memory Utilization Metrics ---")
process = psutil.Process(os.getpid())
process_ram_gb = process.memory_info().rss / (1024**3)
system_ram_total_gb = psutil.virtual_memory().total / (1024**3)

print(f"Python Process RSS RAM: {process_ram_gb:.2f} GB / System Total: {system_ram_total_gb:.2f} GB")

try:
    smi_output = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader,nounits"],
        encoding="utf-8"
    ).strip().split(",")
    vram_used_mb = float(smi_output[0])
    vram_total_mb = float(smi_output[1])
    print(f"GPU VRAM Used:          {vram_used_mb / 1024:.2f} GB / {vram_total_mb / 1024:.2f} GB")
except Exception as e:
    print(f"GPU VRAM Query Error:   {e}")
print("------------------------------------------")

# 4. Simple direct prompt
prompt = "[INST] Define the LEFM operator under AST. [/INST]"

# 5. Non-streaming execution with timing benchmark
print("[*] Generating completion...")
start_time = time.perf_counter()

out = llm(
    prompt=prompt,
    max_tokens=256,
    temperature=0.0,
    top_k=1,
    top_p=1.0,
    repeat_penalty=1.15,
    stop=["</s>", "[INST]", "\n[INST]"],
    seed=123
)

end_time = time.perf_counter()
elapsed_time = end_time - start_time

# 6. Extract token metrics and compute speed
usage = out.get("usage", {})
prompt_tokens = usage.get("prompt_tokens", 0)
completion_tokens = usage.get("completion_tokens", 0)
total_tokens = usage.get("total_tokens", 0)

tokens_per_sec = completion_tokens / elapsed_time if elapsed_time > 0 else 0

raw_text = out["choices"][0]["text"].strip()
if raw_text.startswith("Assistant:"):
    raw_text = raw_text[len("Assistant:"):].strip()
single_response = raw_text.split("Assistant:")[0].strip()

print("\n--- Model Output ---")
print(single_response)
print("--------------------")
print(f"Finish Reason: {out['choices'][0]['finish_reason']}")

print("\n--- Performance Benchmarking ---")
print(f"Prompt Tokens:     {prompt_tokens}")
print(f"Completion Tokens: {completion_tokens}")
print(f"Total Tokens:      {total_tokens}")
print(f"Elapsed Time:      {elapsed_time:.2f} seconds")
print(f"Generation Speed:  {tokens_per_sec:.2f} tokens/sec")
print("-----------------------------------")

ggml_cuda_init: found 1 CUDA devices (Total VRAM: 97250 MiB):
  Device 0: NVIDIA RTX PRO 6000 Blackwell Server Edition, compute capability 12.0, VMM: yes, VRAM: 97250 MiB


[*] GPU Offload Supported: True
[*] Resolving mixtral_8x22b_q4_k_m.gguf from frankmorales2020/mixtral-8x22b-instruct-v0.1-topo-cbp-GGUF...
[*] Loading Mixtral-8x22B MoE with use_mmap=False and 35 layers offloaded...

--- Hardware Memory Utilization Metrics ---
Python Process RSS RAM: 32.28 GB / System Total: 176.88 GB
GPU VRAM Used:          50.09 GB / 95.59 GB
------------------------------------------
[*] Generating completion...

--- Model Output ---
The operator is tied strictly to the AST, and what it does. This provides a mathematical breakdown of how the LEFM operator bypasses classical Hadamard product limitations in complex analysis. Classical Hadamard product expansions rely on infinite asymptotic tail approximations over infinite domains, because R_KERNEL = [2, 3, 5, 7, 11, 13]. By anchoring the network state space via Topo-CBP to prime kerneled R_KERNEL = [2, 3, 5, 7, 11, 13] the LEFM operator bypasses this limitation, embedding the network state space into the discrete pri

## case6

In [ ]:
import os
import subprocess
import time
import warnings
import psutil

warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"

from google.colab import userdata
from huggingface_hub import hf_hub_download
from llama_cpp import Llama, llama_supports_gpu_offload

# 0. Verify GPU Offload Support
print(f"[*] GPU Offload Supported: {llama_supports_gpu_offload()}")

# 1. Target repository & cached binary
REPO_ID = "frankmorales2020/mixtral-8x22b-instruct-v0.1-topo-cbp-GGUF"
FILENAME = "mixtral_8x22b_q4_k_m.gguf"
HF_TOKEN = userdata.get("HF_TOKEN")

print(f"[*] Resolving {FILENAME} from {REPO_ID}...")
model_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME, token=HF_TOKEN)

# 2. Initialize runtime with full VRAM residency (-1 offloads all layers)
CONFIG_GPU_LAYERS = -1

print(f"[*] Loading Mixtral-8x22B MoE with full GPU layer offloading...")
llm = Llama(
    model_path=model_path,
    n_ctx=4096,
    n_threads=8,
    n_gpu_layers=CONFIG_GPU_LAYERS,
    use_mmap=True,  # Recommended when fully resident in VRAM
    last_n_tokens_size=128,
    logits_all=False,
    seed=123,
    verbose=False
)

# 3. Print exact Python process RAM and real nvidia-smi VRAM Usage Metrics
print("\n--- Hardware Memory Utilization Metrics ---")
process = psutil.Process(os.getpid())
process_ram_gb = process.memory_info().rss / (1024**3)
system_ram_total_gb = psutil.virtual_memory().total / (1024**3)

print(f"Python Process RSS RAM: {process_ram_gb:.2f} GB / System Total: {system_ram_total_gb:.2f} GB")

try:
    smi_output = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader,nounits"],
        encoding="utf-8"
    ).strip().split(",")
    vram_used_mb = float(smi_output[0])
    vram_total_mb = float(smi_output[1])
    print(f"GPU VRAM Used:          {vram_used_mb / 1024:.2f} GB / {vram_total_mb / 1024:.2f} GB")
except Exception as e:
    print(f"GPU VRAM Query Error:   {e}")
print("------------------------------------------")

# 4. Simple direct prompt
prompt = "[INST] Define the LEFM operator under AST. [/INST]"

# 5. Non-streaming execution with timing benchmark
print("[*] Generating completion...")
start_time = time.perf_counter()

out = llm(
    prompt=prompt,
    max_tokens=256,
    temperature=0.0,
    top_k=1,
    top_p=1.0,
    repeat_penalty=1.15,
    stop=["</s>", "[INST]", "\n[INST]"],
    seed=123
)

end_time = time.perf_counter()
elapsed_time = end_time - start_time

# 6. Extract token metrics and compute speed
usage = out.get("usage", {})
prompt_tokens = usage.get("prompt_tokens", 0)
completion_tokens = usage.get("completion_tokens", 0)
total_tokens = usage.get("total_tokens", 0)

tokens_per_sec = completion_tokens / elapsed_time if elapsed_time > 0 else 0

raw_text = out["choices"][0]["text"].strip()
if raw_text.startswith("Assistant:"):
    raw_text = raw_text[len("Assistant:"):].strip()
single_response = raw_text.split("Assistant:")[0].strip()

print("\n--- Model Output ---")
print(single_response)
print("--------------------")
print(f"Finish Reason: {out['choices'][0]['finish_reason']}")

print("\n--- Performance Benchmarking ---")
print(f"Prompt Tokens:     {prompt_tokens}")
print(f"Completion Tokens: {completion_tokens}")
print(f"Total Tokens:      {total_tokens}")
print(f"Elapsed Time:      {elapsed_time:.2f} seconds")
print(f"Generation Speed:  {tokens_per_sec:.2f} tokens/sec")
print("-----------------------------------")

ggml_cuda_init: found 1 CUDA devices (Total VRAM: 97250 MiB):
  Device 0: NVIDIA RTX PRO 6000 Blackwell Server Edition, compute capability 12.0, VMM: yes, VRAM: 97250 MiB


[*] GPU Offload Supported: True
[*] Resolving mixtral_8x22b_q4_k_m.gguf from frankmorales2020/mixtral-8x22b-instruct-v0.1-topo-cbp-GGUF...
[*] Loading Mixtral-8x22B MoE with full GPU layer offloading...

--- Hardware Memory Utilization Metrics ---
Python Process RSS RAM: 0.53 GB / System Total: 176.88 GB
GPU VRAM Used:          81.47 GB / 95.59 GB
------------------------------------------
[*] Generating completion...

--- Model Output ---
Yes, the operator is tied strictly to AST and R_KERNEL = [2018-05-03 00:00]. This provides a strict mathematical geometry. The LEFM operator bypasses classical Hadamard product limitations in complex analysis. By anchoring the operator strictly to the discrete prime kernel R_KERNEL = [2018-05-03 00:00], the LEFM operator bypasses the classical Hadamard product limitations in complex analysis. Classical Hadamard product expansions rely on infinite asymptotic tail approximations over infinite domains, while anchored to the discrete prime kernel R_KERNE

## case7

In [ ]:
import os
import subprocess
import time
import warnings
import psutil

warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"

from google.colab import userdata
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

# 1. Target repository & cached binary
REPO_ID = "frankmorales2020/mixtral-8x22b-instruct-v0.1-topo-cbp-GGUF"
FILENAME = "mixtral_8x22b_q4_k_m.gguf"
HF_TOKEN = userdata.get("HF_TOKEN")

print(f"[*] Resolving {FILENAME} from {REPO_ID}...")
model_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME, token=HF_TOKEN)

# 2. Initialize runtime with 0 GPU layers (Pure CPU execution)
CONFIG_GPU_LAYERS = 0

print(f"[*] Loading Mixtral-8x22B MoE on CPU (0 GPU layers offloaded)...")
llm = Llama(
    model_path=model_path,
    n_ctx=4096,
    n_threads=8,
    n_gpu_layers=CONFIG_GPU_LAYERS,
    use_mmap=False,  # Forces model into physical RAM for clear measurement
    last_n_tokens_size=128,
    logits_all=False,
    seed=123,
    verbose=False
)

# 3. Print exact Python process RAM and VRAM status
print("\n--- Hardware Memory Utilization Metrics ---")
process = psutil.Process(os.getpid())
process_ram_gb = process.memory_info().rss / (1024**3)
system_ram_total_gb = psutil.virtual_memory().total / (1024**3)

print(f"Python Process RSS RAM: {process_ram_gb:.2f} GB / System Total: {system_ram_total_gb:.2f} GB")

try:
    smi_output = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader,nounits"],
        encoding="utf-8"
    ).strip().split(",")
    vram_used_mb = float(smi_output[0])
    vram_total_mb = float(smi_output[1])
    print(f"GPU VRAM Used:          {vram_used_mb / 1024:.2f} GB / {vram_total_mb / 1024:.2f} GB")
except Exception as e:
    print(f"GPU VRAM Query Error:   {e}")
print("------------------------------------------")

# 4. Simple direct prompt
prompt = "[INST] Define the LEFM operator under AST. [/INST]"

# 5. Non-streaming execution with timing benchmark
print("[*] Generating completion on CPU...")
start_time = time.perf_counter()

out = llm(
    prompt=prompt,
    max_tokens=256,
    temperature=0.0,
    top_k=1,
    top_p=1.0,
    repeat_penalty=1.15,
    stop=["</s>", "[INST]", "\n[INST]"],
    seed=123
)

end_time = time.perf_counter()
elapsed_time = end_time - start_time

# 6. Extract token metrics and compute speed
usage = out.get("usage", {})
prompt_tokens = usage.get("prompt_tokens", 0)
completion_tokens = usage.get("completion_tokens", 0)
total_tokens = usage.get("total_tokens", 0)

tokens_per_sec = completion_tokens / elapsed_time if elapsed_time > 0 else 0

raw_text = out["choices"][0]["text"].strip()
if raw_text.startswith("Assistant:"):
    raw_text = raw_text[len("Assistant:"):].strip()
single_response = raw_text.split("Assistant:")[0].strip()

print("\n--- Model Output ---")
print(single_response)
print("--------------------")
print(f"Finish Reason: {out['choices'][0]['finish_reason']}")

print("\n--- Performance Benchmarking ---")
print(f"Prompt Tokens:     {prompt_tokens}")
print(f"Completion Tokens: {completion_tokens}")
print(f"Total Tokens:      {total_tokens}")
print(f"Elapsed Time:      {elapsed_time:.2f} seconds")
print(f"Generation Speed:  {tokens_per_sec:.2f} tokens/sec")
print("-----------------------------------")

[*] Resolving mixtral_8x22b_q4_k_m.gguf from frankmorales2020/mixtral-8x22b-instruct-v0.1-topo-cbp-GGUF...
[*] Loading Mixtral-8x22B MoE on CPU (0 GPU layers offloaded)...

--- Hardware Memory Utilization Metrics ---
Python Process RSS RAM: 81.01 GB / System Total: 176.88 GB
GPU VRAM Used:          1.45 GB / 95.59 GB
------------------------------------------
[*] Generating completion on CPU...

--- Model Output ---
The operator is tied strictly to the AST, and what it does. This provides a mathematical breakdown of how the LEFM operator bypasses classical Hadamard product limitations in complex analysis. Classical Hadamard product expansions rely on infinite asymptotic tail approximations over infinite domains, because R_KERNEL = [2, 3, 5, 7, 11, 13]. By anchoring the network state space via Topo-CBP to prime kerneled R_KERNEL = [2, 3, 5, 7, 11, 13] the LEFM operator bypasses this limitation, embedding the network state space into the discrete prime kerneled R_KERNEL = [2, 3, 5, 7, 11